# Visualization Gallery

A showcase of different visualization techniques using matplotlib, seaborn, and plotly.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("✅ Libraries loaded!")

## Generate Sample Data

In [ ]:
# Create diverse dataset
n = 500
dates = pd.date_range('2023-01-01', periods=n, freq='D')

df = pd.DataFrame({
    'date': dates,
    'value': np.cumsum(np.random.randn(n)) + 100,
    'category': np.random.choice(['A', 'B', 'C', 'D'], n),
    'x': np.random.randn(n),
    'y': np.random.randn(n),
    'size': np.random.randint(10, 100, n),
    'group': np.random.choice(['Group 1', 'Group 2', 'Group 3'], n)
})

df['y_corr'] = df['x'] * 2 + np.random.randn(n) * 0.5

print(f"Dataset shape: {df.shape}")
df.head()

## 1. Matplotlib Visualizations

In [ ]:
# Line plot with multiple series
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Line plot
for cat in df['category'].unique():
    data = df[df['category'] == cat]
    axes[0, 0].plot(data['date'], data['value'].rolling(7).mean(), label=cat, alpha=0.7)
axes[0, 0].set_title('Time Series by Category (7-day MA)')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Value')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Histogram
axes[0, 1].hist(df['value'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Value Distribution')
axes[0, 1].set_xlabel('Value')
axes[0, 1].set_ylabel('Frequency')

# Scatter plot
for group in df['group'].unique():
    data = df[df['group'] == group]
    axes[1, 0].scatter(data['x'], data['y_corr'], label=group, alpha=0.5)
axes[1, 0].set_title('Scatter Plot with Groups')
axes[1, 0].set_xlabel('X')
axes[1, 0].set_ylabel('Y')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Bar plot
category_means = df.groupby('category')['value'].mean()
axes[1, 1].bar(category_means.index, category_means.values, edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Average Value by Category')
axes[1, 1].set_xlabel('Category')
axes[1, 1].set_ylabel('Average Value')

plt.tight_layout()
plt.show()

## 2. Seaborn Visualizations

In [ ]:
# Seaborn plot grid
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Violin plot
sns.violinplot(data=df, x='category', y='value', ax=axes[0, 0])
axes[0, 0].set_title('Value Distribution by Category')

# Box plot
sns.boxplot(data=df, x='group', y='value', hue='category', ax=axes[0, 1])
axes[0, 1].set_title('Value by Group and Category')
axes[0, 1].tick_params(axis='x', rotation=45)

# Heatmap (correlation)
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1, 0], 
            square=True, linewidths=1)
axes[1, 0].set_title('Correlation Matrix')

# Joint plot (using separate figure)
sns.scatterplot(data=df, x='x', y='y_corr', hue='category', ax=axes[1, 1], alpha=0.6)
axes[1, 1].set_title('X vs Y by Category')

plt.tight_layout()
plt.show()

In [ ]:
# Pair plot (creates its own figure)
sample_df = df[['value', 'x', 'y_corr', 'category']].sample(200)
sns.pairplot(sample_df, hue='category', diag_kind='kde', corner=True)
plt.suptitle('Pair Plot', y=1.01)
plt.show()

## 3. Plotly Interactive Visualizations

In [ ]:
# Interactive time series
fig = px.line(df, x='date', y='value', color='category',
              title='Interactive Time Series',
              labels={'value': 'Value', 'date': 'Date'})
fig.update_layout(hovermode='x unified')
fig.show()

In [ ]:
# Interactive scatter with size
fig = px.scatter(df, x='x', y='y_corr', color='category', size='size',
                 title='Interactive Scatter Plot with Size',
                 hover_data=['group', 'value'])
fig.show()

In [ ]:
# 3D scatter plot
fig = px.scatter_3d(df.sample(200), x='x', y='y_corr', z='value',
                    color='category', size='size',
                    title='3D Scatter Plot')
fig.show()

In [ ]:
# Sunburst chart
summary = df.groupby(['group', 'category']).size().reset_index(name='count')
fig = px.sunburst(summary, path=['group', 'category'], values='count',
                  title='Hierarchical Distribution')
fig.show()

In [ ]:
# Subplots with plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Box Plot', 'Histogram', 'Pie Chart', 'Bar Chart'),
    specs=[[{'type': 'box'}, {'type': 'histogram'}],
           [{'type': 'pie'}, {'type': 'bar'}]]
)

# Box plot
for cat in df['category'].unique():
    fig.add_trace(go.Box(y=df[df['category']==cat]['value'], name=cat),
                  row=1, col=1)

# Histogram
fig.add_trace(go.Histogram(x=df['value'], nbinsx=30, name='Value'),
              row=1, col=2)

# Pie chart
category_counts = df['category'].value_counts()
fig.add_trace(go.Pie(labels=category_counts.index, values=category_counts.values),
              row=2, col=1)

# Bar chart
group_means = df.groupby('group')['value'].mean()
fig.add_trace(go.Bar(x=group_means.index, y=group_means.values, name='Avg Value'),
              row=2, col=2)

fig.update_layout(height=800, showlegend=True, title_text="Multiple Chart Types")
fig.show()

## 4. Advanced Visualizations

In [ ]:
# Animated scatter plot
df_sample = df.sample(100).copy()
df_sample['time'] = pd.cut(df_sample['date'], bins=10, labels=range(10))

fig = px.scatter(df_sample, x='x', y='y_corr', 
                 animation_frame='time',
                 color='category', size='size',
                 range_x=[-3, 3], range_y=[-6, 6],
                 title='Animated Scatter Plot')
fig.show()

In [ ]:
# Contour plot
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.sqrt(X**2 + Y**2))

fig = go.Figure(data=go.Contour(z=Z, x=x, y=y, colorscale='Viridis'))
fig.update_layout(title='Contour Plot Example')
fig.show()

## Summary

This notebook showcases various visualization techniques:
- **Matplotlib**: Static, publication-quality plots
- **Seaborn**: Statistical visualizations with beautiful defaults
- **Plotly**: Interactive, web-ready visualizations

Choose the right tool for your needs:
- Use **matplotlib** for fine-grained control and publication
- Use **seaborn** for statistical analysis and exploration
- Use **plotly** for interactive dashboards and presentations